[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yryo1005/Deep_Variation_Information_Bottleneck_training/blob/master/ex005_mnist_vib_2d.ipynb)

> **実行する前に，ランタイムを GPU に変更してください．**  
> Google Colab ではメニューの **ランタイム → ランタイムのタイプを変更** を開き，ハードウェア アクセラレータを **GPU** に設定して保存します．CPU のままだと学習に時間がかかります．

# 実験005: Deep VIB の 2 次元特徴マップの可視化

本ノートブックでは，潜在次元を 2 にした Deep Variational Information Bottleneck（Deep VIB）を MNIST で学習し，潜在空間を可視化する．

この講座では，次の順でプログラムを積み上げる．

1. NN による MNIST 分類（交差エントロピー誤差）
2. Autoencoder による MNIST の再構成
3. Variational Autoencoder による MNIST の再構成
4. Deep Variational Information Bottleneck による MNIST 分類
5. **本実験**: 2 次元特徴マップの可視化
6. 敵対的学習（ノイズ最適化）
7. Deep VIB + Brier スコアによる MNIST 分類
8. Deep VIB + クラス間重み付き CCE による MNIST 分類

学習ループの分割は実験004と同じである．本実験での変更点は，潜在次元を 2 にし，学習後に潜在空間を可視化することである．


## 1. ライブラリ


In [ ]:
import json
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm.auto import tqdm


## 2. 再現性とログ


In [ ]:
def set_seed(seed=0):
    """実験の再現性を担保するため，乱数シードを固定する．

    Args:
        seed (int): 固定するシード値．デフォルトは 0．

    Returns:
        None
    """
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.use_deterministic_algorithms(True)


class ResultLogger:
    """学習中の指標（Loss，Accuracy，KL など）を記録し，JSON として保存・読み込みするクラス．

    Args:
        target_path (str | None): 既存ログ JSON のパス．指定時は初期化時に読み込む．

    Returns:
        なし（インスタンスを生成する）
    """

    def __init__(self, target_path=None):
        self.names = None
        self.history = {}
        if target_path:
            self.load(target_path)

    def set_names(self, *names):
        """記録する指標名を登録する．

        Args:
            *names (str): 指標名．例: "train_loss", "train_acc"．

        Returns:
            None
        """
        if self.names:
            raise RuntimeError("指標名はすでに登録されています．")
        self.names = list(names)
        for name in self.names:
            if name not in self.history:
                self.history[name] = []

    def __call__(self, *values):
        """1 エポック分の指標値を履歴へ追加する．

        Args:
            *values (float | int): set_names で登録した順の値．

        Returns:
            None
        """
        if self.names is None:
            raise RuntimeError("先に set_names で指標名を登録してください．")
        if len(values) != len(self.names):
            raise RuntimeError("値の数が登録済み指標名の数と一致しません．")
        for name, value in zip(self.names, values):
            self.history[name].append(value)

    def save(self, target_path):
        """履歴を JSON ファイルとして保存する．

        Args:
            target_path (str): 保存先パス．

        Returns:
            None
        """
        with open(target_path, "w") as f:
            json.dump(self.history, f, indent=4)

    def load(self, target_path):
        """JSON ファイルから履歴を読み込む．

        Args:
            target_path (str): 読み込む JSON のパス．

        Returns:
            None
        """
        with open(target_path, "r") as f:
            data = json.load(f)
        self.history = data
        self.names = list(data.keys())

    def __getitem__(self, key):
        """指定した指標の履歴リストを取得する．

        Args:
            key (str): 指標名．

        Returns:
            history (list): 指標の履歴．存在しない場合は空リスト．
        """
        return self.history.get(key, [])


## 3. 実験設定

`outputs/ex005_mnist_vib_2d/{最適化手法}/{学習率}_{バッチサイズ}/{seed}/`

本実験はサンプルのため，シードは 1 つ（`seed=0`）とする．潜在次元は可視化のため 2 とする．


In [ ]:
EXPERIMENT_NAME = "ex005_mnist_vib_2d"

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name == EXPERIMENT_NAME:
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

DATASET_DIR = PROJECT_ROOT / "datasets" / EXPERIMENT_NAME / "standard"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / EXPERIMENT_NAME

EPOCHS = 10
SEEDS = [0]
OPTIMIZER_NAMES = ["Adam"]
LEARNING_RATES = [0.001]
BATCH_SIZES = [128]
LATENT_DIM = 2
BETA = 0.001

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATASET_DIR : {DATASET_DIR}")
print(f"OUTPUT_ROOT : {OUTPUT_ROOT}")
print(f"LATENT_DIM  : {LATENT_DIM}")
print(f"BETA        : {BETA}")


## 4. モデルの定義

実験004の Deep VIB と同じ構造である．違いは潜在次元が 2 である点だけである．

- Encoder: 784 -> 256 -> 128 -> ($\mu$, $\log\sigma^2$)，各 2 次元
- Reparameterization: $z = \mu + \epsilon \odot \exp(0.5\log\sigma^2)$
- Classifier: 2 -> 10（クラスロジット）

2 次元の $\mu$ をそのまま散布図の座標として使う．


In [ ]:
class MNISTVIB(nn.Module):
    """MNIST を分類する Deep Variational Information Bottleneck（2 次元潜在空間）．

    Encoder は 784 -> 256 -> 128 ののち，潜在平均 mu と log 分散 logvar を出力する．
    Classifier は潜在ベクトル z から 10 クラスのロジットを計算する．

    Args:
        latent_dim (int): 潜在空間の次元数．可視化のため 2 を用いる．
    """

    def __init__(self, latent_dim=2):
        super().__init__()
        self.latent_dim = latent_dim
        self.enc1 = nn.Linear(784, 256)
        self.enc2 = nn.Linear(256, 128)
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        self.fc_out = nn.Linear(latent_dim, 10)

    def encode(self, x):
        """入力画像から潜在分布 q(z|x) のパラメータ mu, logvar を計算する．

        Args:
            x (torch.Tensor): 入力画像．形状は (N, 1, 28, 28) または (N, 784)．

        Returns:
            mu (torch.Tensor): 潜在平均．形状は (N, latent_dim)．
            logvar (torch.Tensor): 潜在 log 分散．形状は (N, latent_dim)．
        """
        x = x.view(x.size(0), -1)
        x = F.relu(self.enc1(x))
        x = F.relu(self.enc2(x))
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        """Reparameterization Trick で潜在ベクトル z をサンプリングする．

        Args:
            mu (torch.Tensor): 潜在平均．形状は (N, latent_dim)．
            logvar (torch.Tensor): 潜在 log 分散．形状は (N, latent_dim)．

        Returns:
            z (torch.Tensor): サンプルされた潜在ベクトル．形状は (N, latent_dim)．
        """
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        return z

    def classify(self, z):
        """潜在ベクトルからクラスロジットを計算する．

        Args:
            z (torch.Tensor): 潜在ベクトル．形状は (N, latent_dim)．

        Returns:
            logits (torch.Tensor): クラスロジット．形状は (N, 10)．
        """
        logits = self.fc_out(z)
        return logits

    def forward(self, x):
        """Deep VIB の順伝播を行い，ロジットと分布パラメータを返す．

        Args:
            x (torch.Tensor): 入力画像．形状は (N, 1, 28, 28) または (N, 784)．

        Returns:
            outputs (dict): キー "logits", "mu", "logvar" を持つ辞書．
                logits (torch.Tensor): クラスロジット，形状 (N, 10)．
                mu (torch.Tensor): 潜在平均，形状 (N, latent_dim)．
                logvar (torch.Tensor): 潜在 log 分散，形状 (N, latent_dim)．
        """
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        logits = self.classify(z)
        return {"logits": logits, "mu": mu, "logvar": logvar}

    def predict(self, x):
        """潜在平均 mu を用いて決定的にクラスロジットを計算する．

        Args:
            x (torch.Tensor): 入力画像．形状は (N, 1, 28, 28) または (N, 784)．

        Returns:
            logits (torch.Tensor): クラスロジット．形状は (N, 10)．
        """
        mu, _ = self.encode(x)
        return self.classify(mu)


def load_model(ModelClass, weight_path=None, seed=0, latent_dim=2):
    """モデルをインスタンス化し，必要なら学習済み重みを読み込む．

    Args:
        ModelClass (type): torch.nn.Module を継承したモデルクラス．
        weight_path (str | None): 学習済み重み (.pth) のパス．None なら初期値を使う．
        seed (int): パラメータ初期化用の乱数シード．
        latent_dim (int): 潜在空間の次元数．

    Returns:
        model (torch.nn.Module): インスタンス化されたモデル．
    """
    set_seed(seed)
    model = ModelClass(latent_dim=latent_dim)
    if weight_path is not None:
        state_dict = torch.load(weight_path, map_location="cpu")
        model.load_state_dict(state_dict)
    return model


## 5. データセットと DataLoader

MNIST は公式の学習 60,000 枚 / テスト 10,000 枚に分割されている．画素値は `ToTensor()` により $[0, 1]$ に正規化する．DataLoader のシャッフルは `torch.Generator` で決定論的にする．


In [ ]:
def load_dataloader(seed=0, batch_size=128):
    """MNIST の学習用・検証用 DataLoader を作成する．

    Args:
        seed (int): データの並びを固定する乱数シード．
        batch_size (int): ミニバッチサイズ．

    Returns:
        train_dataloader (torch.utils.data.DataLoader): 学習用 DataLoader．
            各バッチは画像 (N, 1, 28, 28) とラベル (N,) のタプル．
        test_dataloader (torch.utils.data.DataLoader): 検証用 DataLoader．
            各バッチの形状は学習用と同じ．
    """
    set_seed(seed)
    DATASET_DIR.mkdir(parents=True, exist_ok=True)

    transform = transforms.ToTensor()
    train_dataset = datasets.MNIST(
        root=str(DATASET_DIR),
        train=True,
        download=True,
        transform=transform,
    )
    test_dataset = datasets.MNIST(
        root=str(DATASET_DIR),
        train=False,
        download=True,
        transform=transform,
    )

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_dataloader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        generator=generator,
        num_workers=0,
    )
    test_dataloader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )
    return train_dataloader, test_dataloader


### データの確認

学習用 DataLoader から 1 バッチを取り出し，画像の形状とラベルの例を表示する．


In [ ]:
preview_loader, _ = load_dataloader(seed=0, batch_size=8)
preview_images, preview_labels = next(iter(preview_loader))
print(f"images shape: {tuple(preview_images.shape)}")
print(f"labels shape: {tuple(preview_labels.shape)}")
print(f"labels      : {preview_labels.tolist()}")

fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
for i, ax in enumerate(axes):
    ax.imshow(preview_images[i, 0], cmap="gray")
    ax.set_title(f"label={preview_labels[i].item()}")
    ax.axis("off")
plt.tight_layout()
plt.show()


## 6. 誤差関数と評価関数

実験004と同じ VIB 損失を用いる．

$$
\mathcal{L} = \mathcal{L}_{\mathrm{CE}} + \beta \mathcal{L}_{\mathrm{KL}}
$$

評価指標は Accuracy と KL とする．


In [ ]:
def kl_divergence(mu, logvar):
    """q(z|x)=N(mu, diag(exp(logvar))) と p(z)=N(0, I) の KL ダイバージェンスを計算する．

    Args:
        mu (torch.Tensor): 潜在平均．形状は (N, latent_dim)．
        logvar (torch.Tensor): 潜在 log 分散．形状は (N, latent_dim)．

    Returns:
        kl (torch.Tensor): 1 サンプルあたりの平均 KL．形状は ()．
    """
    kl = -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1))
    return kl


def loss_func(outputs, teacher_signals):
    """交差エントロピーと KL ダイバージェンスの加重和（VIB 損失）を計算する．

    Args:
        outputs (dict): モデル出力．キー "logits", "mu", "logvar" を持つ．
        teacher_signals (torch.Tensor): クラスラベル．形状は (N,)，dtype は long．

    Returns:
        loss (torch.Tensor): 1 データあたりの平均 VIB 損失．形状は ()．
    """
    logits = outputs["logits"]
    mu = outputs["mu"]
    logvar = outputs["logvar"]
    ce_loss = F.cross_entropy(logits, teacher_signals, reduction="mean")
    kl_loss = kl_divergence(mu, logvar)
    loss = ce_loss + BETA * kl_loss
    return loss


def metrics_func(outputs, teacher_signals):
    """モデル出力と教師信号から Accuracy と KL を計算する．

    Args:
        outputs (dict): モデル出力．キー "logits", "mu", "logvar" を持つ．
        teacher_signals (torch.Tensor): クラスラベル．形状は (N,)．

    Returns:
        metrics_to_value (dict): キー "acc", "kl" を持つ辞書．
    """
    logits = outputs["logits"]
    mu = outputs["mu"]
    logvar = outputs["logvar"]
    pred_labels = logits.argmax(dim=1)
    acc = (pred_labels == teacher_signals).float().mean().item()
    kl = kl_divergence(mu, logvar).item()
    return {"acc": acc, "kl": kl}


## 7. 学習ループ

学習は `iteration`，`epoch`，`train` の 3 段に分ける．実験004と同じである．


In [ ]:
def iteration(model, inputs, teacher_signals, optimizer=None):
    """1 ミニバッチを学習または検証する．

    Args:
        model (torch.nn.Module): 学習 / 検証対象のモデル．
        inputs (torch.Tensor): 入力画像．形状は (N, 1, 28, 28)．
        teacher_signals (torch.Tensor): クラスラベル．形状は (N,)．
        optimizer (torch.optim.Optimizer | None): 最適化手法．None なら検証モード．

    Returns:
        metrics_to_value (dict): "loss", "acc", "kl" をキーとする 1 データあたりの平均値の辞書．
    """
    if optimizer is None:
        with torch.no_grad():
            outputs = model(inputs)
            loss = loss_func(outputs, teacher_signals)
    else:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_func(outputs, teacher_signals)
        loss.backward()
        optimizer.step()

    metrics_to_value = metrics_func(outputs, teacher_signals)
    metrics_to_value["loss"] = loss.item()
    return metrics_to_value


def epoch(model, dataloader, optimizer=None):
    """DataLoader 全件を 1 周し，1 データあたりの平均指標を返す．

    Args:
        model (torch.nn.Module): 学習 / 検証対象のモデル．
        dataloader (torch.utils.data.DataLoader): 入力とラベルの DataLoader．
        optimizer (torch.optim.Optimizer | None): 最適化手法．None なら検証モード．

    Returns:
        metrics_to_value (dict): エポック全体の 1 データあたり平均（"loss", "acc", "kl"）．
    """
    if optimizer is None:
        model.eval()
    else:
        model.train()

    device = next(model.parameters()).device
    sum_metrics = {}
    n_total = 0

    progress = tqdm(dataloader, leave=False)
    for inputs, teacher_signals in progress:
        inputs = inputs.to(device)
        teacher_signals = teacher_signals.to(device)

        batch_metrics = iteration(model, inputs, teacher_signals, optimizer)
        batch_size = inputs.size(0)
        n_total += batch_size

        for name, value in batch_metrics.items():
            sum_metrics[name] = sum_metrics.get(name, 0.0) + value * batch_size

        average_metrics = {name: total / n_total for name, total in sum_metrics.items()}
        progress.set_postfix({name: f"{value:.4f}" for name, value in average_metrics.items()})

    metrics_to_value = {name: total / n_total for name, total in sum_metrics.items()}
    return metrics_to_value


In [ ]:
def build_optimizer(model, optimizer_name, lr):
    """最適化手法の名前から Optimizer を生成する．

    Args:
        model (torch.nn.Module): パラメータを更新するモデル．
        optimizer_name (str): 最適化手法名．"Adam" または "SGD"．
        lr (float): 学習率．

    Returns:
        optimizer (torch.optim.Optimizer): 生成された Optimizer．
    """
    if optimizer_name == "Adam":
        return torch.optim.Adam(model.parameters(), lr=lr)
    if optimizer_name == "SGD":
        return torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    raise ValueError(f"未対応の最適化手法です: {optimizer_name}")


def train(
    target_dir,
    ModelClass,
    load_dataloader,
    epochs,
    batch_size,
    seed=0,
    lr=0.001,
    optimizer_name="Adam",
    latent_dim=2,
):
    """モデルを学習し，最良重みとログを保存する．

    Args:
        target_dir (str): 結果の保存先ディレクトリ．
        ModelClass (type): 学習するモデルクラス．
        load_dataloader (callable): seed と batch_size から DataLoader を返す関数．
        epochs (int): 学習エポック数．
        batch_size (int): ミニバッチサイズ．
        seed (int): 乱数シード．
        lr (float): 学習率．
        optimizer_name (str): 最適化手法名．
        latent_dim (int): 潜在空間の次元数．

    Returns:
        None
    """
    os.makedirs(target_dir, exist_ok=True)
    train_dataloader, test_dataloader = load_dataloader(seed=seed, batch_size=batch_size)
    model = load_model(ModelClass, weight_path=None, seed=seed, latent_dim=latent_dim)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    optimizer = build_optimizer(model, optimizer_name, lr)

    logger = ResultLogger()
    logger.set_names("train_loss", "train_acc", "train_kl", "val_loss", "val_acc", "val_kl")

    train_metrics = epoch(model, train_dataloader, optimizer=None)
    val_metrics = epoch(model, test_dataloader, optimizer=None)
    logger(
        train_metrics["loss"],
        train_metrics["acc"],
        train_metrics["kl"],
        val_metrics["loss"],
        val_metrics["acc"],
        val_metrics["kl"],
    )
    print(
        f"[epoch 0/{epochs}] "
        f"train_loss={train_metrics['loss']:.4f} train_acc={train_metrics['acc']:.4f} train_kl={train_metrics['kl']:.4f} "
        f"val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['acc']:.4f} val_kl={val_metrics['kl']:.4f}"
    )

    best_val_acc = val_metrics["acc"]
    torch.save(model.state_dict(), os.path.join(target_dir, "best_model.pth"))

    for epoch_idx in range(1, epochs + 1):
        train_metrics = epoch(model, train_dataloader, optimizer=optimizer)
        val_metrics = epoch(model, test_dataloader, optimizer=None)
        logger(
            train_metrics["loss"],
            train_metrics["acc"],
            train_metrics["kl"],
            val_metrics["loss"],
            val_metrics["acc"],
            val_metrics["kl"],
        )
        print(
            f"[epoch {epoch_idx}/{epochs}] "
            f"train_loss={train_metrics['loss']:.4f} train_acc={train_metrics['acc']:.4f} train_kl={train_metrics['kl']:.4f} "
            f"val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['acc']:.4f} val_kl={val_metrics['kl']:.4f}"
        )

        if val_metrics["acc"] > best_val_acc:
            best_val_acc = val_metrics["acc"]
            torch.save(model.state_dict(), os.path.join(target_dir, "best_model.pth"))

        logger.save(os.path.join(target_dir, "log.json"))

    logger.save(os.path.join(target_dir, "log.json"))
    print(f"best val_acc={best_val_acc:.4f}")
    print(f"saved: {target_dir}")


## 8. 実行


In [ ]:
print(f"device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

for optimizer_name in OPTIMIZER_NAMES:
    for lr in LEARNING_RATES:
        for batch_size in BATCH_SIZES:
            for seed in SEEDS:
                hyperparam_name = f"{lr}_{batch_size}"
                target_dir = OUTPUT_ROOT / optimizer_name / hyperparam_name / str(seed)
                log_path = target_dir / "log.json"
                weight_path = target_dir / "best_model.pth"

                if log_path.exists() and weight_path.exists():
                    print(f"skip: {target_dir}")
                    continue

                print(f"train: {target_dir}")
                train(
                    target_dir=str(target_dir),
                    ModelClass=MNISTVIB,
                    load_dataloader=load_dataloader,
                    epochs=EPOCHS,
                    batch_size=batch_size,
                    seed=seed,
                    lr=lr,
                    optimizer_name=optimizer_name,
                    latent_dim=LATENT_DIM,
                )


## 9. 学習曲線の確認

保存したログから Loss，Accuracy，KL の推移を描画する．


In [ ]:
log_path = OUTPUT_ROOT / OPTIMIZER_NAMES[0] / f"{LEARNING_RATES[0]}_{BATCH_SIZES[0]}" / str(SEEDS[0]) / "log.json"
logger = ResultLogger(str(log_path))
epochs_axis = list(range(len(logger["train_loss"])))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].plot(epochs_axis, logger["train_loss"], label="train")
axes[0].plot(epochs_axis, logger["val_loss"], label="val")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("loss")
axes[0].set_title("VIB Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_axis, logger["train_acc"], label="train")
axes[1].plot(epochs_axis, logger["val_acc"], label="val")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("accuracy")
axes[1].set_title("Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs_axis, logger["train_kl"], label="train")
axes[2].plot(epochs_axis, logger["val_kl"], label="val")
axes[2].set_xlabel("epoch")
axes[2].set_ylabel("kl")
axes[2].set_title("KL Divergence")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"final train_loss = {logger['train_loss'][-1]:.4f}")
print(f"final val_loss   = {logger['val_loss'][-1]:.4f}")
print(f"final train_acc  = {logger['train_acc'][-1]:.4f}")
print(f"final val_acc    = {logger['val_acc'][-1]:.4f}")
print(f"best val_acc     = {max(logger['val_acc']):.4f}")
print(f"final train_kl   = {logger['train_kl'][-1]:.4f}")
print(f"final val_kl     = {logger['val_kl'][-1]:.4f}")


## 10. 2 次元特徴マップの可視化

最良モデルで検証データを Encoder に通し，潜在平均 $\mu = (\mu_1, \mu_2)$ を収集する．クラスごとに色を分けて散布図を描く．クラスが分かれていれば，2 次元ボトルネックに識別情報が残っている．

決定領域（分離領域）は，データ点そのものではなく **潜在平面を格子に切って分類器へ通す** ことで求める．手順は次のとおりである．

1. 検証データの $\mu$ の最小・最大から余白 $0.5$ を足し，描画範囲 $[x_{\min}, x_{\max}]\times[y_{\min}, y_{\max}]$ を決める
2. 各軸を $n_{\mathrm{grid}}=200$ 点で等間隔に区切り，`meshgrid` で格子点 $z_{i,j}\in\mathbb{R}^{2}$ を作る（合計 $200\times 200$ 点）
3. 各格子点を分類器 `classify` に通し，予測クラス $\hat{y}_{i,j}=\arg\max_k \ell_k(z_{i,j})$ を得る
4. `contourf` で格子の予測クラスを色面として塗り，その上に検証データの $\mu$ を散布する

したがって背景色は「その座標の $z$ を分類器に入れたときにどのクラスになるか」であり，点の密度から推定した境界ではない．分類器は線形（$\ell = W z + b$）なので，領域の境界は直線に近い．


In [ ]:
def collect_latent_means(model, dataloader):
    """DataLoader 全件の潜在平均 mu，正解ラベル，予測ラベルを収集する．

    Args:
        model (torch.nn.Module): encode と classify を持つモデル．
        dataloader (torch.utils.data.DataLoader): 画像とラベルの DataLoader．

    Returns:
        mus (np.ndarray): 潜在平均．形状は (N, 2)．
        labels (np.ndarray): 正解ラベル．形状は (N,)．
        preds (np.ndarray): 予測ラベル．形状は (N,)．
    """
    model.eval()
    device = next(model.parameters()).device
    mu_list = []
    label_list = []
    pred_list = []

    progress = tqdm(dataloader, leave=False)
    for inputs, teacher_signals in progress:
        inputs = inputs.to(device)
        with torch.no_grad():
            mu, _ = model.encode(inputs)
            logits = model.classify(mu)
            pred_labels = logits.argmax(dim=1)
        mu_list.append(mu.cpu().numpy())
        label_list.append(teacher_signals.numpy())
        pred_list.append(pred_labels.cpu().numpy())

    mus = np.concatenate(mu_list, axis=0)
    labels = np.concatenate(label_list, axis=0)
    preds = np.concatenate(pred_list, axis=0)
    return mus, labels, preds


def plot_latent_scatter(mus, labels, save_path):
    """潜在平均 mu をクラス色の散布図として保存する．

    Args:
        mus (np.ndarray): 潜在平均．形状は (N, 2)．
        labels (np.ndarray): クラスラベル．形状は (N,)．
        save_path (pathlib.Path): 保存先パス．

    Returns:
        None
    """
    fig, ax = plt.subplots(figsize=(5, 5))
    cmap = plt.get_cmap("tab10")
    for class_id in range(10):
        mask = labels == class_id
        ax.scatter(
            mus[mask, 0],
            mus[mask, 1],
            s=8,
            alpha=0.5,
            color=cmap(class_id),
            label=str(class_id),
        )
    ax.set_xlabel(r"$\mu_1$")
    ax.set_ylabel(r"$\mu_2$")
    ax.set_title("2D latent means")
    ax.legend(markerscale=2, fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_aspect("equal", adjustable="datalim")
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


def plot_decision_regions(model, mus, labels, save_path, n_grid=200):
    """2 次元潜在平面の分類境界と mu の散布を重ねて保存する．

    Args:
        model (torch.nn.Module): classify を持つモデル．
        mus (np.ndarray): 潜在平均．形状は (N, 2)．
        labels (np.ndarray): クラスラベル．形状は (N,)．
        save_path (pathlib.Path): 保存先パス．
        n_grid (int): 各軸の格子点数．

    Returns:
        None
    """
    margin = 0.5
    x_min, x_max = mus[:, 0].min() - margin, mus[:, 0].max() + margin
    y_min, y_max = mus[:, 1].min() - margin, mus[:, 1].max() + margin
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, n_grid),
        np.linspace(y_min, y_max, n_grid),
    )
    grid = np.stack([xx.ravel(), yy.ravel()], axis=1)

    device = next(model.parameters()).device
    model.eval()
    with torch.no_grad():
        z_grid = torch.from_numpy(grid).float().to(device)
        logits = model.classify(z_grid)
        grid_preds = logits.argmax(dim=1).cpu().numpy().reshape(xx.shape)

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.contourf(xx, yy, grid_preds, levels=np.arange(11) - 0.5, cmap="tab10", alpha=0.25)
    cmap = plt.get_cmap("tab10")
    for class_id in range(10):
        mask = labels == class_id
        ax.scatter(
            mus[mask, 0],
            mus[mask, 1],
            s=8,
            alpha=0.5,
            color=cmap(class_id),
            label=str(class_id),
        )
    ax.set_xlabel(r"$\mu_1$")
    ax.set_ylabel(r"$\mu_2$")
    ax.set_title("Decision regions")
    ax.legend(markerscale=2, fontsize=8)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_aspect("equal", adjustable="box")
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


In [ ]:
weight_path = OUTPUT_ROOT / OPTIMIZER_NAMES[0] / f"{LEARNING_RATES[0]}_{BATCH_SIZES[0]}" / str(SEEDS[0]) / "best_model.pth"
_, test_loader = load_dataloader(seed=SEEDS[0], batch_size=BATCH_SIZES[0])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = load_model(MNISTVIB, weight_path=str(weight_path), seed=SEEDS[0], latent_dim=LATENT_DIM)
model = model.to(device)

mus, labels, preds = collect_latent_means(model, test_loader)
print(f"mus shape   : {mus.shape}")
print(f"labels shape: {labels.shape}")
print(f"test acc    : {(preds == labels).mean():.4f}")

figure_dir = OUTPUT_ROOT / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)

plot_latent_scatter(mus, labels, figure_dir / "latent_scatter.png")
plot_decision_regions(model, mus, labels, figure_dir / "decision_regions.png")
print(f"saved: {figure_dir / 'latent_scatter.png'}")
print(f"saved: {figure_dir / 'decision_regions.png'}")
